# Task 3 — Cleaning Data

**Objective:** Demonstrate professional-level data cleaning by taking a deliberately messy dataset and systematically transforming it into a clean, analysis-ready dataset.

**Dataset:** Titanic passenger dataset (a deliberately corrupted practice copy is used so that missing values, duplicates, inconsistent formats, data-type issues and outliers can be demonstrated).

**Tech Stack:** Python, pandas, NumPy, Jupyter Notebook

### Cleaning workflow
1. Load and inspect the messy dataset
2. Produce a data quality report
3. Handle missing values
4. Remove duplicate rows
5. Standardise inconsistent values
6. Correct data types
7. Detect and handle outliers using IQR
8. Produce a before-vs-after summary
9. Save the cleaned dataset as CSV


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

df = pd.read_csv("Titanic_Messy.csv")
print("Dataset shape:", df.shape)
df.head()


/home/oai/.config/matplotlib is not a writable directory


Matplotlib created a temporary cache directory at /tmp/matplotlib-55faje8z because there was an issue with the default path (/home/oai/.config/matplotlib); it is highly recommended to set the MPLCONFIGDIR environment variable to a writable directory, in particular to speed up the import of Matplotlib and to better support multiprocessing.


Dataset shape: (903, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",Male,22.0,1,0,A/5 21171,7.25,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",Female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",Female,26.0,0,0,STON/O2. 3101282,7.925,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",Female,35.0,1,0,113803,53.1,C123,S
4,5,0,3,"Allen, Mr. William Henry",Male,35.0,0,0,373450,8.05,NaN,S


## 1. Initial Inspection

In [2]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nUnique values in categorical columns:")
print("Sex:", df["Sex"].dropna().astype(str).str.strip().unique())
print("Embarked:", df["Embarked"].dropna().astype(str).str.strip().unique())

print("\nNumeric summary:")
print(df[["Age", "Fare", "SibSp", "Parch"]].describe().T)


Shape: (903, 12)

Data types:
PassengerId     int64
Survived        int64
Pclass          int64
Name           object
Sex            object
Age            object
SibSp           int64
Parch           int64
Ticket         object
Fare           object
Cabin          object
Embarked       object
dtype: object

Missing values:
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            187
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          697
Embarked         6
dtype: int64

Duplicate rows: 12

Unique values in categorical columns:
Sex: ['Male' 'Female' 'F' 'M' 'female' 'f' 'male' 'MALE']
Embarked: ['S' 'C' 'Q' 'c' 's' 'q']

Numeric summary:
       count      mean       std  min  25%  50%  75%  max
SibSp  903.0  0.521595  1.097936  0.0  0.0  0.0  1.0  8.0
Parch  903.0  0.380952  0.803921  0.0  0.0  0.0  0.0  6.0


## 2. Data Quality Report

The report records the main quality problems before cleaning: null counts, duplicate count, data types, and potential numeric anomalies.

In [3]:
quality_report = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "null_count": df.isna().sum(),
    "null_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True)
})

quality_report


,dtype,null_count,null_percent,unique_values
PassengerId,int64,0,0.00,891
Survived,int64,0,0.00,2
Pclass,int64,0,0.00,3
Name,object,0,0.00,891
Sex,object,0,0.00,10
Age,object,187,20.71,88
SibSp,int64,0,0.00,7
Parch,int64,0,0.00,7
Ticket,object,0,0.00,681
Fare,object,0,0.00,265


## 3. Missing Value Handling

**Strategy:**
- `Age`: fill missing values with the median because age is numeric and can contain skew/outliers.
- `Embarked`: fill with the mode because it is a categorical variable.
- `Cabin`: drop the column because most cabin values are missing, making reliable imputation difficult for this cleaning exercise.
- Other columns: retain when no missing values are present.

In [4]:
before_missing = df.isna().sum()

# Convert numeric columns first so corrupted numeric strings become NaN.
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Fare"] = pd.to_numeric(df["Fare"], errors="coerce")
df["PassengerId"] = pd.to_numeric(df["PassengerId"], errors="coerce")

df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Cabin has a very high missing rate in this dataset.
df = df.drop(columns=["Cabin"])

print("Missing values after missing-value treatment:")
print(df.isna().sum())


Missing values after missing-value treatment:
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           7
Embarked       0
dtype: int64


## 4. Duplicate Removal

Duplicate records can bias analysis, so exact duplicate rows are identified and removed. The number removed is documented.

In [5]:
duplicate_count_before = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
duplicate_count_after = df.duplicated().sum()

print("Duplicate rows before:", duplicate_count_before)
print("Duplicate rows removed:", duplicate_count_before - duplicate_count_after)
print("Duplicate rows after:", duplicate_count_after)


Duplicate rows before: 12
Duplicate rows removed: 12
Duplicate rows after: 0


## 5. Standardisation of Inconsistent Formatting

Examples such as `male`, `MALE`, `M`, `F`, and extra spaces are converted to consistent `Male`/`Female` values. Embarkation codes are standardised to uppercase `C`, `Q`, and `S`.

In [6]:
# Standardise Sex
def clean_sex(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip().lower()
    if value in ["m", "male"]:
        return "Male"
    if value in ["f", "female"]:
        return "Female"
    return np.nan

df["Sex"] = df["Sex"].apply(clean_sex)

# Standardise Embarked
df["Embarked"] = (
    df["Embarked"]
    .astype("string")
    .str.strip()
    .str.upper()
    .replace({"": pd.NA})
)

df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

print("Sex values:", df["Sex"].unique())
print("Embarked values:", df["Embarked"].unique())


Sex values: ['Male' 'Female']
Embarked values: <StringArray>
['S', 'C', 'Q']
Length: 3, dtype: string


## 6. Correct Data Types

The final schema is explicitly set so that IDs are integers, binary/count fields are integers, age/fare are numeric, and categorical fields use consistent string/category types.

In [7]:
df["PassengerId"] = pd.to_numeric(df["PassengerId"], errors="coerce").astype("Int64")
df["Survived"] = pd.to_numeric(df["Survived"], errors="coerce").astype("Int64")
df["Pclass"] = pd.to_numeric(df["Pclass"], errors="coerce").astype("Int64")
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["SibSp"] = pd.to_numeric(df["SibSp"], errors="coerce").astype("Int64")
df["Parch"] = pd.to_numeric(df["Parch"], errors="coerce").astype("Int64")
df["Fare"] = pd.to_numeric(df["Fare"], errors="coerce")

df["Name"] = df["Name"].astype("string")
df["Sex"] = df["Sex"].astype("string")
df["Ticket"] = df["Ticket"].astype("string")
df["Embarked"] = df["Embarked"].astype("string")

print(df.dtypes)


PassengerId             Int64
Survived                Int64
Pclass                  Int64
Name           string[python]
Sex            string[python]
Age                   float64
SibSp                   Int64
Parch                   Int64
Ticket         string[python]
Fare                  float64
Embarked       string[python]
dtype: object


## 7. Outlier Detection — IQR Method

The Interquartile Range (IQR) method is used for numeric variables. Values below `Q1 - 1.5×IQR` or above `Q3 + 1.5×IQR` are flagged.

For this dataset, extreme **Fare** values are capped at the IQR limits instead of deleting passengers, because the values may represent legitimate expensive tickets.

In [8]:
def iqr_bounds(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - 1.5 * iqr, q3 + 1.5 * iqr

outlier_summary = []

for col in ["Age", "Fare", "SibSp", "Parch"]:
    lower, upper = iqr_bounds(df[col].dropna())
    mask = (df[col] < lower) | (df[col] > upper)
    outlier_summary.append({
        "Column": col,
        "Lower Bound": round(lower, 2),
        "Upper Bound": round(upper, 2),
        "Outliers": int(mask.sum())
    })

outlier_summary = pd.DataFrame(outlier_summary)
outlier_summary


,Column,Lower Bound,Upper Bound,Outliers
0,Age,2.50,54.50,64
1,Fare,-26.76,65.66,115
2,SibSp,-1.50,2.50,46
3,Parch,0.00,0.00,213


In [9]:
# Cap Fare outliers instead of removing rows.
fare_lower, fare_upper = iqr_bounds(df["Fare"].dropna())
fare_outliers_before = int(((df["Fare"] < fare_lower) | (df["Fare"] > fare_upper)).sum())

df["Fare"] = df["Fare"].clip(lower=fare_lower, upper=fare_upper)

fare_outliers_after = int(((df["Fare"] < fare_lower) | (df["Fare"] > fare_upper)).sum())

print("Fare outliers before capping:", fare_outliers_before)
print("Fare outliers after capping:", fare_outliers_after)
print("Fare limits used:", round(fare_lower, 2), "to", round(fare_upper, 2))


Fare outliers before capping: 115
Fare outliers after capping: 0
Fare limits used: -26.76 to 65.66


## 8. Final Quality Check

In [10]:
final_quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "null_count": df.isna().sum(),
    "null_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(dropna=True)
})

print("Final shape:", df.shape)
print("Final duplicate count:", df.duplicated().sum())
print("\nFinal quality report:")
final_quality


Final shape: (891, 11)
Final duplicate count: 0

Final quality report:


,dtype,null_count,null_percent,unique_values
PassengerId,Int64,0,0.00,891
Survived,Int64,0,0.00,2
Pclass,Int64,0,0.00,3
Name,string,0,0.00,891
Sex,string,0,0.00,2
Age,float64,0,0.00,87
SibSp,Int64,0,0.00,7
Parch,Int64,0,0.00,7
Ticket,string,0,0.00,681
Fare,float64,7,0.79,213


## 9. Before vs After Summary

This table documents the main data-quality changes required by the task.

In [11]:
before_rows = len(pd.read_csv("Titanic_Messy.csv"))
before_cols = len(pd.read_csv("Titanic_Messy.csv").columns)
before_nulls = int(pd.read_csv("Titanic_Messy.csv").isna().sum().sum())
before_dups = int(pd.read_csv("Titanic_Messy.csv").duplicated().sum())

summary = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Total null cells",
        "Duplicate rows",
        "Remaining duplicate rows",
        "Remaining null cells"
    ],
    "Before": [
        before_rows,
        before_cols,
        before_nulls,
        before_dups,
        "-",
        "-"
    ],
    "After": [
        len(df),
        len(df.columns),
        int(df.isna().sum().sum()),
        "-",
        int(df.duplicated().sum()),
        int(df.isna().sum().sum())
    ]
})

summary


,Metric,Before,After
0,Rows,903,891
1,Columns,12,11
2,Total null cells,890,7
3,Duplicate rows,12,-
4,Remaining duplicate rows,-,0
5,Remaining null cells,-,7


## 10. Save the Cleaned Dataset

In [12]:
output_file = "Titanic_Cleaned.csv"
df.to_csv(output_file, index=False)

print(f"Cleaned dataset saved as: {output_file}")
print(f"Rows: {len(df)} | Columns: {len(df.columns)}")


Cleaned dataset saved as: Titanic_Cleaned.csv
Rows: 891 | Columns: 11


## Conclusion

The deliberately messy Titanic dataset was transformed into an analysis-ready dataset by:

- identifying missing values, duplicates and inconsistent formats;
- imputing numeric and categorical missing values with justified methods;
- removing duplicate rows;
- standardising categorical values;
- correcting data types;
- detecting numeric outliers using the IQR method;
- capping extreme Fare values rather than deleting potentially valid passenger records;
- documenting the before-vs-after data quality.

The cleaned file is saved as `Titanic_Cleaned.csv`.
